# Table Generating Functions in Databricks

Table generating functions (also called **table-valued functions** or **generator functions**) transform a **single row** into **multiple rows** or expand complex data structures into tabular format.

## Key Concepts

* **Input**: One row with complex data (arrays, structs, JSON)
* **Output**: Multiple rows with simpler/flattened data
* **Usage**: Typically used with `LATERAL VIEW` for inline expansion

## Common Table Generating Functions

| Function | Purpose |
|----------|----------|
| `explode(array)` | One row per array element |
| `posexplode(array)` | Explode with position/index |
| `explode_outer(array)` | Explode, preserving NULL/empty arrays |
| `inline(array<struct>)` | Expand array of structs to rows |
| `stack(n, col1, col2, ...)` | Pivot columns to rows |
| `json_tuple(json, fields...)` | Extract multiple JSON fields |

## When to Use

✅ **Unnesting arrays** — One row per array element  
✅ **Flattening nested structures** — Convert structs to columns  
✅ **Pivoting data** — Transform columns to rows  
✅ **JSON extraction** — Extract multiple fields efficiently  

Let's explore each function with examples!

In [0]:
%sql
-- Create sample data with arrays for demonstrations
CREATE OR REPLACE TEMP VIEW users_with_arrays AS
SELECT 1 as user_id, 'Alice' as name, array('apple', 'banana', 'cherry') as favorite_fruits, array('reading', 'hiking') as hobbies
UNION ALL
SELECT 2 as user_id, 'Bob' as name, array('orange') as favorite_fruits, array('gaming', 'cooking', 'coding') as hobbies
UNION ALL
SELECT 3 as user_id, 'Charlie' as name, array() as favorite_fruits, array('swimming') as hobbies
UNION ALL
SELECT 4 as user_id, 'Diana' as name, NULL as favorite_fruits, array('yoga', 'painting') as hobbies;

SELECT * FROM users_with_arrays;

In [0]:
%sql
-- EXPLODE: Creates one row per array element
-- Syntax: explode(array_column)

-- Method 1: Using CTE (Common Table Expression)
WITH exploded AS (
  SELECT 
    user_id,
    name,
    explode(favorite_fruits) as fruit
  FROM users_with_arrays
)
SELECT * FROM exploded;

In [0]:
%sql
-- Method 2: Using LATERAL VIEW (inline expansion)
-- LATERAL VIEW creates a virtual table and joins it with the original row

SELECT 
  user_id,
  name,
  fruit
FROM users_with_arrays
LATERAL VIEW explode(favorite_fruits) fruits_table AS fruit;

-- Note: Rows with NULL or empty arrays are excluded (Charlie and Diana)

In [0]:
%sql
-- EXPLODE_OUTER: Like explode(), but preserves rows with NULL or empty arrays
-- Returns NULL for the exploded column when array is NULL/empty

SELECT 
  user_id,
  name,
  fruit
FROM users_with_arrays
LATERAL VIEW OUTER explode(favorite_fruits) fruits_table AS fruit;

-- Key difference: Charlie (empty array) and Diana (NULL) are now included with fruit = NULL

In [0]:
%sql
-- POSEXPLODE: Like explode(), but also returns the position (0-based index) of each element
-- Useful when the order of elements matters

SELECT 
  user_id,
  name,
  pos,
  fruit
FROM users_with_arrays
LATERAL VIEW posexplode(favorite_fruits) fruits_table AS pos, fruit
ORDER BY user_id, pos;

In [0]:
%sql
-- Create sample data with array of structs for INLINE demonstration
CREATE OR REPLACE TEMP VIEW orders AS
SELECT 
  1 as order_id,
  array(
    named_struct('product', 'Laptop', 'quantity', 2, 'price', 1200.00),
    named_struct('product', 'Mouse', 'quantity', 1, 'price', 25.00)
  ) as items
UNION ALL
SELECT 
  2 as order_id,
  array(
    named_struct('product', 'Keyboard', 'quantity', 3, 'price', 75.00)
  ) as items;

SELECT * FROM orders;

In [0]:
%sql
-- INLINE: Explodes an array of structs into rows, with each struct field becoming a column
-- Perfect for nested arrays where each element has multiple attributes

SELECT 
  order_id,
  product,
  quantity,
  price,
  quantity * price as total
FROM orders
LATERAL VIEW inline(items) items_table;

-- Each struct in the items array becomes a separate row with individual columns

In [0]:
%sql
-- STACK: Transforms multiple columns into multiple rows (unpivot operation)
-- Syntax: stack(n, col1_name, col1_value, col2_name, col2_value, ...)
-- where n is the number of output rows per input row

-- Sample data
CREATE OR REPLACE TEMP VIEW sales_quarterly AS
SELECT 'Product A' as product, 100 as Q1, 150 as Q2, 200 as Q3, 180 as Q4
UNION ALL
SELECT 'Product B' as product, 80 as Q1, 90 as Q2, 110 as Q3, 120 as Q4;

-- Unpivot quarterly sales into rows
SELECT 
  product,
  quarter,
  sales
FROM sales_quarterly
LATERAL VIEW stack(
  4,
  'Q1', Q1,
  'Q2', Q2,
  'Q3', Q3,
  'Q4', Q4
) unpivoted AS quarter, sales;

In [0]:
%sql
select * from sales_quarterly

In [0]:
%sql
-- Multiple LATERAL VIEWs can be chained to explode multiple arrays
-- Creates a CARTESIAN product: every combination of elements from both arrays

SELECT 
  user_id,
  name,
  fruit,
  hobby
FROM users_with_arrays
LATERAL VIEW explode(favorite_fruits) fruits AS fruit
LATERAL VIEW explode(hobbies) hobbies AS hobby
ORDER BY user_id;

-- Result: Alice (3 fruits × 2 hobbies = 6 rows), Bob (1 fruit × 3 hobbies = 3 rows)

In [0]:
%sql
-- Direct comparison: explode() vs explode_outer()
-- explode(): Excludes NULL/empty arrays
-- explode_outer(): Includes NULL/empty arrays with NULL values

SELECT 'EXPLODE' as function_type, user_id, name, fruit
FROM users_with_arrays
LATERAL VIEW explode(favorite_fruits) t AS fruit

UNION ALL

SELECT 'EXPLODE_OUTER' as function_type, user_id, name, fruit
FROM users_with_arrays
LATERAL VIEW OUTER explode(favorite_fruits) t AS fruit
ORDER BY user_id, function_type;

## Table Generating Functions - Complete Reference

---

### Function Comparison

| Function | Input | Output | NULL/Empty Handling | Use Case |
|----------|-------|--------|---------------------|----------|
| **`explode(array)`** | Array | One row per element | Excludes NULL/empty | Basic array unnesting |
| **`explode_outer(array)`** | Array | One row per element | Includes with NULL | Preserve all rows |
| **`posexplode(array)`** | Array | (pos, value) per element | Excludes NULL/empty | Track element position |
| **`posexplode_outer(array)`** | Array | (pos, value) per element | Includes with NULL | Position + preserve rows |
| **`inline(array<struct>)`** | Array of structs | One row per struct, fields as columns | N/A | Flatten nested objects |
| **`inline_outer(array<struct>)`** | Array of structs | One row per struct | Includes NULL/empty | Flatten + preserve rows |
| **`stack(n, ...)`** | Columns | n rows with pivoted values | N/A | Unpivot/melt columns |
| **`json_tuple(json, ...)`** | JSON string | Columns for each field | N/A | Extract JSON fields |

---

### Syntax Patterns

#### Using CTE (Common Table Expression)
```sql
WITH exploded AS (
  SELECT id, explode(array_col) as value
  FROM table
)
SELECT * FROM exploded;
```

#### Using LATERAL VIEW (Inline)
```sql
SELECT id, value
FROM table
LATERAL VIEW explode(array_col) t AS value;
```

#### Chaining Multiple LATERAL VIEWs
```sql
SELECT id, val1, val2
FROM table
LATERAL VIEW explode(array1) t1 AS val1
LATERAL VIEW explode(array2) t2 AS val2;
```

---

### Key Differences

#### EXPLODE vs EXPLODE_OUTER
* **`explode()`**: Filters out rows where array is NULL or empty
* **`explode_outer()`**: Keeps rows with NULL or empty arrays (exploded value = NULL)
* **When to use OUTER**: When you need to preserve all original rows regardless of array content

#### CTE vs LATERAL VIEW
* **CTE**: More readable for complex multi-step logic; easier debugging
* **LATERAL VIEW**: More concise for simple transformations; inline expansion
* **Performance**: Generally equivalent; choose based on readability

---

### Common Patterns

#### 1. Flatten Array of Structs
```sql
SELECT id, field1, field2
FROM table
LATERAL VIEW inline(array_of_structs) t;
```

#### 2. Unpivot Columns to Rows
```sql
SELECT id, metric, value
FROM table
LATERAL VIEW stack(3, 'metric1', col1, 'metric2', col2, 'metric3', col3) t AS metric, value;
```

#### 3. Explode with Position Tracking
```sql
SELECT id, pos, value
FROM table
LATERAL VIEW posexplode(array_col) t AS pos, value;
```

#### 4. Cartesian Product of Two Arrays
```sql
SELECT id, val1, val2
FROM table
LATERAL VIEW explode(array1) t1 AS val1
LATERAL VIEW explode(array2) t2 AS val2;
```

---

### Best Practices

✅ **Use `explode_outer()` when you can't afford to lose rows** (auditing, completeness)  
✅ **Use `inline()` for array of structs** — cleaner than explode + field access  
✅ **Use `posexplode()` when order matters** — e.g., ranking, sequence analysis  
✅ **Use `stack()` for unpivoting** — better than UNION ALL for many columns  
✅ **Chain LATERAL VIEWs carefully** — creates Cartesian products (can explode row counts)  

❌ **Avoid nested `explode()` in SELECT** — use LATERAL VIEW or CTEs instead  
❌ **Don't explode without understanding cardinality** — can create millions of rows  
❌ **Don't use `explode()` when `explode_outer()` is needed** — silent data loss

---

### Performance Tips

* Table generating functions are generally efficient in Spark
* Watch out for Cartesian products when chaining multiple LATERAL VIEWs
* Consider filtering BEFORE exploding to reduce intermediate data size
* Use `LIMIT` during development to avoid accidentally creating huge result sets